In [1]:
import pandas as pd
import numpy as np
import pickle

In [2]:
df=pd.read_csv('penguins.csv')
df

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007
...,...,...,...,...,...,...,...,...
339,Chinstrap,Dream,55.8,19.8,207.0,4000.0,male,2009
340,Chinstrap,Dream,43.5,18.1,202.0,3400.0,female,2009
341,Chinstrap,Dream,49.6,18.2,193.0,3775.0,male,2009
342,Chinstrap,Dream,50.8,19.0,210.0,4100.0,male,2009


In [3]:
df.dropna(inplace=True)
df

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,year
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,male,2007
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,female,2007
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,female,2007
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,female,2007
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,male,2007
...,...,...,...,...,...,...,...,...
339,Chinstrap,Dream,55.8,19.8,207.0,4000.0,male,2009
340,Chinstrap,Dream,43.5,18.1,202.0,3400.0,female,2009
341,Chinstrap,Dream,49.6,18.2,193.0,3775.0,male,2009
342,Chinstrap,Dream,50.8,19.0,210.0,4100.0,male,2009


In [4]:
df.describe()

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,year
count,333.000000,333.000000,333.000000,333.000000,333.000000
mean,43.992793,17.164865,200.966967,4207.057057,2008.042042
std,5.468668,1.969235,14.015765,805.215802,0.812944
min,32.100000,13.100000,172.000000,2700.000000,2007.000000
25%,39.500000,15.600000,190.000000,3550.000000,2007.000000
50%,44.500000,17.300000,197.000000,4050.000000,2008.000000
75%,48.600000,18.700000,213.000000,4775.000000,2009.000000
max,59.600000,21.500000,231.000000,6300.000000,2009.000000


In [5]:
df.shape

(333, 8)

In [6]:
df=df.drop(columns=['island','species'])
df['sex']=df['sex'].astype('category')
df['sex']=df['sex'].cat.codes

In [7]:
for x in ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']:
    df[x]=(df[x]-df[x].min())/(df[x].max()-df[x].min())

In [8]:
y = df['sex']
x = df.drop(columns=['sex'])

In [9]:
np.random.seed(1200)
ratio = 0.20
total_rows = df.shape[0]
test_size = int(total_rows*ratio)

In [10]:
x_test = x[0:test_size]
x_train = x[test_size:]
y_test = y[0:test_size]
y_train = y[test_size:]

C:\Users\svais\AppData\Local\Temp\ipykernel_21680\1508126261.py:3: FutureWarning: The behavior of `series[i:j]` with an integer-dtype index is deprecated. In a future version, this will be treated as *label-based* indexing, consistent with e.g. `series[i]` lookups. To retain the old behavior, use `series.iloc[i:j]`. To get the future behavior, use `series.loc[i:j]`.
  y_test = y[0:test_size]
C:\Users\svais\AppData\Local\Temp\ipykernel_21680\1508126261.py:4: FutureWarning: The behavior of `series[i:j]` with an integer-dtype index is deprecated. In a future version, this will be treated as *label-based* indexing, consistent with e.g. `series[i]` lookups. To retain the old behavior, use `series.iloc[i:j]`. To get the future behavior, use `series.loc[i:j]`.
  y_train = y[test_size:]


In [11]:
x_train.shape

(267, 5)

In [12]:
x_test.shape

(66, 5)

In [13]:
y_train.shape

(267,)

In [14]:
y_test.shape

(66,)

In [24]:

class LogitRegression:
   
    def __init__(self,learning_rate,iterations):
        self.learning_rate=learning_rate
        self.iterations=iterations
    
    def sigmoid(self,z):
        S=1/(1+np.exp(-z))
        return S
    
    def cost(self,x,y):
        c=np.dot(x,self.weights) 
        h=self.sigmoid(c)
        return np.mean(-y*np.log(h)-(1-y)*np.log(1-h))
    
    def gradient_descent(self,x,y):
        z=np.dot(x,self.weights)+self.bias
        predict=self.sigmoid(z)
        d=predict-y
        d_weights=(1/len(y))*np.dot(x.transpose(),d)
        d_bias=(1/len(y))*np.sum(d)
        return d_weights,d_bias
    
    def fit(self,x,y):
        self.loss=[]
        self.weights=np.random.uniform(0,1,x.shape[1])
        self.bias=0
        for i in range(0,self.iterations):
            dw,db=self.gradient_descent(x, y) 
            self.bias-=self.learning_rate*db
            self.weights-=self.learning_rate*dw
            self.loss.append(self.cost(x,y))     
        return self.weights,self.loss
    
    def predict(self,x):
        z=np.dot(x,self.weights)+self.bias
        h=1/(1+np.exp(-z))
        y_pred=np.where(h >= 0.5, 1, 0)
        return y_pred


In [25]:
model=LogitRegression(1e-3,100000)
weights,loss=model.fit(x_train, y_train)

C:\Users\svais\AppData\Local\Temp\ipykernel_21680\939185758.py:14: RuntimeWarning: divide by zero encountered in log
  return np.mean(-y*np.log(h)-(1-y)*np.log(1-h))
C:\Users\svais\AppData\Local\Temp\ipykernel_21680\939185758.py:8: RuntimeWarning: overflow encountered in exp
  S=1/(1+np.exp(-z))


In [28]:
y_pred=model.predict(x_test)
accuracy=np.mean(y_pred == y_test)
print("Accuracy:", accuracy)

Accuracy: 0.5


In [27]:
with open('penguins_model.pkl', 'wb') as f:
    pickle.dump(weights, f)